## 1. Exploración del trial corpus
Análisis exploratorio de los datos de la competición MentalRiskES 2026
para entender la estructura y distribución de las conversaciones.

In [18]:
import json
import os
import pandas as pd
from google.colab import drive

# Monto Drive
drive.mount('/content/drive')
RUTA = "/content/drive/MyDrive/TFG"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
# Compruebo qué hay en la carpeta data
print(os.listdir(f"{RUTA}/data"))

['.gitkeep', 'task2_trial.zip', 'task2_trial', 'dataset_evaluacion.csv']


In [20]:
# Compruebo qué hay dentro de task2_trial
print(os.listdir(f"{RUTA}/data/task2_trial"))

['data']


In [21]:
print(os.listdir(f"{RUTA}/data/task2_trial/data"))

['round_19.json', 'round_16.json', 'round_18.json', 'round_11.json', 'round_4.json', 'round_9.json', 'round_13.json', 'round_1.json', 'round_2.json', 'round_10.json', 'round_8.json', 'round_17.json', 'round_6.json', 'round_7.json', 'round_14.json', 'round_5.json', 'round_15.json', 'round_3.json', 'round_12.json']


In [22]:
# Ruta correcta
ruta_data = f"{RUTA}/data/task2_trial/data"

# Cargo todas las rondas ordenadas
rondas = []
for archivo in sorted(os.listdir(ruta_data)):
    if archivo.endswith(".json"):
        with open(f"{ruta_data}/{archivo}", "r", encoding="utf-8") as f:
            datos = json.load(f)
            rondas.append(datos)

print(f"Total de rondas: {len(rondas)}")
print(f"Sesiones en ronda 1: {len(rondas[0])}")
print(f"IDs de sesiones: {list(rondas[0].keys())}")

Total de rondas: 19
Sesiones en ronda 1: 1
IDs de sesiones: ['trial']


In [23]:
# Construyo la conversación completa de la sesión trial
print("CONVERSACION COMPLETA DE LA SESION TRIAL")
print("=" * 60)

for i, ronda in enumerate(rondas):
    sesion = ronda["trial"]
    print(f"\nRONDA {i+1}")
    print(f"Paciente: {sesion['patient_input']}")
    print(f"Opcion 1: {sesion['option_1']}")
    print(f"Opcion 2: {sesion['option_2']}")
    print(f"Opcion 3: {sesion['option_3']}")
    print("-" * 60)

CONVERSACION COMPLETA DE LA SESION TRIAL

RONDA 1
Paciente: Hola. La verdad es que hoy me siento fatal. No sé ni por dónde empezar… Llevo semanas así, pero hoy ha sido el punto de quiebre. No puedo concentrarme en nada, la ansiedad me está carcomiendo y siento que voy a explotar. Necesitaba hablar con alguien, de verdad.
Opcion 1: Entiendo que estás pasando por un momento muy difícil. Es normal que te sientas así dada tu historia. Quiero que sepas que estás en un lugar seguro y que puedo escucharte sin juzgarte. ¿Podrías decirme más sobre lo que te ha llevado a este punto de quiebre? ¿Qué ha sido lo que ha desencadenado esta sensación de ansiedad y no poder concentrarte?
Opcion 2: Entiendo que hoy te sientas abrumado y que la ansiedad esté haciendo que te sientas como si estuvieras a punto de explotar. Es normal que te sientas así, especialmente cuando llevas semanas en este estado. ¿Qué te gustaría que hicieras hoy, aunque sea algo pequeño, para empezar a moverte hacia algo que te imp

In [24]:
# Analisis estadistico del trial corpus
datos_analisis = []

for i, ronda in enumerate(rondas):
    sesion = ronda["trial"]
    datos_analisis.append({
        "ronda": i + 1,
        "longitud_paciente": len(sesion["patient_input"].split()),
        "longitud_opcion_1": len(sesion["option_1"].split()),
        "longitud_opcion_2": len(sesion["option_2"].split()),
        "longitud_opcion_3": len(sesion["option_3"].split()),
    })

df = pd.DataFrame(datos_analisis)
df["opcion_mas_larga"] = df[["longitud_opcion_1", "longitud_opcion_2", "longitud_opcion_3"]].idxmax(axis=1)

print("ESTADISTICAS DEL TRIAL CORPUS")
print("=" * 40)
print(f"Numero de rondas: {len(df)}")
print(f"\nLongitud media mensaje paciente: {df['longitud_paciente'].mean():.1f} palabras")
print(f"Longitud media opcion 1: {df['longitud_opcion_1'].mean():.1f} palabras")
print(f"Longitud media opcion 2: {df['longitud_opcion_2'].mean():.1f} palabras")
print(f"Longitud media opcion 3: {df['longitud_opcion_3'].mean():.1f} palabras")
print(f"\nDistribucion de opciones mas largas:")
print(df["opcion_mas_larga"].value_counts())

ESTADISTICAS DEL TRIAL CORPUS
Numero de rondas: 19

Longitud media mensaje paciente: 40.1 palabras
Longitud media opcion 1: 78.6 palabras
Longitud media opcion 2: 79.6 palabras
Longitud media opcion 3: 81.5 palabras

Distribucion de opciones mas largas:
opcion_mas_larga
longitud_opcion_3    10
longitud_opcion_2     7
longitud_opcion_1     2
Name: count, dtype: int64


Esto significa que el baseline de respuesta más larga elegiría option_3 la mayoría de las veces. Sin etiquetas no sabemos si eso es bueno o malo, pero ya tenemos el análisis.

In [25]:
# Descargo los datasets externos
!pip install datasets -q

from datasets import load_dataset

# Counsel Chat
print("Descargando Counsel Chat...")
counsel_chat = load_dataset("nbertagnolli/counsel-chat")
print(f"Counsel Chat cargado: {len(counsel_chat['train'])} ejemplos")
print(f"Columnas: {counsel_chat['train'].column_names}")

Descargando Counsel Chat...


Repo card metadata block was not found. Setting CardData to empty.


Counsel Chat cargado: 2775 ejemplos
Columnas: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views']


Perfecto, 2775 ejemplos. Ahora exploramos su estructura para entender cómo adaptarlo a tu tarea:

In [26]:
# Exploro la estructura de Counsel Chat
df_counsel = counsel_chat["train"].to_pandas()

print("ESTRUCTURA DE COUNSEL CHAT")
print("=" * 40)
print(f"Total ejemplos: {len(df_counsel)}")
print(f"\nPreguntas unicas: {df_counsel['questionID'].nunique()}")
print(f"Topics disponibles: {df_counsel['topic'].nunique()}")
print(f"\nDistribucion de upvotes:")
print(df_counsel['upvotes'].describe())
print(f"\nEjemplo de pregunta:")
print(df_counsel['questionText'].iloc[0])
print(f"\nEjemplo de respuesta:")
print(df_counsel['answerText'].iloc[0][:300])

ESTRUCTURA DE COUNSEL CHAT
Total ejemplos: 2775

Preguntas unicas: 940
Topics disponibles: 31

Distribucion de upvotes:
count    2775.000000
mean        0.519640
std         1.042825
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        12.000000
Name: upvotes, dtype: float64

Ejemplo de pregunta:
I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.
   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?

Ejemplo de respuesta:
It is very common for people to have multiple issues that they want to (and need to) address in counseling.  I have had clients ask that same question and through more exploration, there is often an underlying fear that they  "can't be helped" or that they will

Perfecto. Ahora entiendo la estructura. Cada pregunta tiene varias respuestas de distintos terapeutas con upvotes. La respuesta con más upvotes es la "correcta".
Vamos a transformarlo al formato de tu tarea:

In [27]:
# Transformo Counsel Chat al formato de la tarea:
# dado un mensaje del paciente + 3 opciones → respuesta correcta

import random

# Agrupo por pregunta y me quedo con las que tienen al menos 3 respuestas
grupos = df_counsel.groupby("questionID")
dataset_evaluacion = []

for question_id, grupo in grupos:
    if len(grupo) >= 3:
        # Ordeno por upvotes descendente
        grupo = grupo.sort_values("upvotes", ascending=False)

        # La correcta es la que tiene más upvotes
        correcta = grupo.iloc[0]["answerText"]
        otras = grupo.iloc[1:]["answerText"].tolist()

        # Elijo 2 respuestas incorrectas al azar
        otras_2 = random.sample(otras, min(2, len(otras)))

        # Mezclo las 3 opciones aleatoriamente
        opciones = [correcta] + otras_2
        random.shuffle(opciones)

        # La etiqueta es el índice de la correcta
        etiqueta = opciones.index(correcta)

        dataset_evaluacion.append({
            "patient_input": grupo.iloc[0]["questionText"],
            "option_1": opciones[0],
            "option_2": opciones[1],
            "option_3": opciones[2],
            "correct": f"option_{etiqueta + 1}",
            "topic": grupo.iloc[0]["topic"]
        })

df_eval = pd.DataFrame(dataset_evaluacion)
print(f"Dataset de evaluacion creado: {len(df_eval)} ejemplos")
print(f"\nDistribucion de respuestas correctas:")
print(df_eval["correct"].value_counts())
print(f"\nTopics:")
print(df_eval["topic"].value_counts().head(10))

Dataset de evaluacion creado: 261 ejemplos

Distribucion de respuestas correctas:
correct
option_2    89
option_1    87
option_3    85
Name: count, dtype: int64

Topics:
topic
anxiety              41
depression           39
intimacy             23
relationships        21
parenting            19
self-esteem          12
trauma               12
lgbtq                11
family-conflict      10
behavioral-change     8
Name: count, dtype: int64


In [33]:
# Guardo el dataset de evaluacion
ruta_guardado = f"{RUTA}/data/dataset_evaluacion.csv"
df_eval.to_csv(ruta_guardado, index=False)
print(f"Dataset guardado en: {ruta_guardado}")
print(f"\nPrimeros ejemplos:")
print(df_eval[["patient_input", "correct", "topic"]].head(3))

Dataset guardado en: /content/drive/MyDrive/TFG/data/dataset_evaluacion.csv

Primeros ejemplos:
                                       patient_input   correct       topic
0  I have so many issues to address. I have a his...  option_1  depression
1  I have been diagnosed with general anxiety and...  option_2  depression
2  There are many people willing to lovingly prov...  option_2  depression


## 2. Implementación de baselines
Sistemas de selección de respuesta simples que sirven como punto de comparación.

In [30]:
import random

def baseline_aleatorio(row):
    return random.choice(["option_1", "option_2", "option_3"])

def baseline_mas_larga(row):
    longitudes = {
        "option_1": len(str(row["option_1"]).split()),
        "option_2": len(str(row["option_2"]).split()),
        "option_3": len(str(row["option_3"]).split())
    }
    return max(longitudes, key=longitudes.get)

def baseline_lexica(row):
    palabras_paciente = set(str(row["patient_input"]).lower().split())
    coincidencias = {
        "option_1": len(palabras_paciente & set(str(row["option_1"]).lower().split())),
        "option_2": len(palabras_paciente & set(str(row["option_2"]).lower().split())),
        "option_3": len(palabras_paciente & set(str(row["option_3"]).lower().split()))
    }
    return max(coincidencias, key=coincidencias.get)

print("Funciones de baselines definidas correctamente")

Funciones de baselines definidas correctamente


In [31]:
# Elimino filas con valores nulos
df_eval = df_eval.dropna(subset=["option_1", "option_2", "option_3", "patient_input"])
print(f"Ejemplos tras limpiar nulos: {len(df_eval)}")

# Vuelvo a aplicar los baselines
df_eval["pred_aleatorio"] = df_eval.apply(baseline_aleatorio, axis=1)
df_eval["pred_mas_larga"] = df_eval.apply(baseline_mas_larga, axis=1)
df_eval["pred_lexica"] = df_eval.apply(baseline_lexica, axis=1)

# Calculo accuracy
acc_aleatorio = (df_eval["pred_aleatorio"] == df_eval["correct"]).mean()
acc_mas_larga = (df_eval["pred_mas_larga"] == df_eval["correct"]).mean()
acc_lexica = (df_eval["pred_lexica"] == df_eval["correct"]).mean()

print("\nRESULTADOS DE LOS BASELINES")
print("=" * 40)
print(f"Baseline aleatorio:           {acc_aleatorio:.3f} ({acc_aleatorio*100:.1f}%)")
print(f"Baseline respuesta mas larga: {acc_mas_larga:.3f} ({acc_mas_larga*100:.1f}%)")
print(f"Baseline coincidencia lexica: {acc_lexica:.3f} ({acc_lexica*100:.1f}%)")

Ejemplos tras limpiar nulos: 240

RESULTADOS DE LOS BASELINES
Baseline aleatorio:           0.312 (31.2%)
Baseline respuesta mas larga: 0.379 (37.9%)
Baseline coincidencia lexica: 0.367 (36.7%)


Esto ya es muy interesante para tu TFG:

- El aleatorio da ~33% como era de esperar (1 de 3 opciones)
- La respuesta más larga y la léxica mejoran ligeramente, pero no mucho
- Esto sugiere que la tarea es difícil y necesita comprensión semántica real

### 2.4 Baseline LLM zero-shot
Selección de respuesta usando un modelo de lenguaje sin contexto adicional.

In [34]:
!pip install groq -q
from groq import Groq

GROQ_API_KEY = "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"
cliente_groq = Groq(api_key=GROQ_API_KEY)
MODELO = "llama-3.3-70b-versatile"

def baseline_llm_zeroshot(row):
    prompt = f"""You are a clinical psychology expert.
Select the most appropriate therapist response for the following patient message.

PATIENT MESSAGE:
{row['patient_input']}

OPTIONS:
1. {row['option_1']}
2. {row['option_2']}
3. {row['option_3']}

Reply ONLY with "option_1", "option_2" or "option_3"."""

    respuesta = cliente_groq.chat.completions.create(
        model=MODELO,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    texto = respuesta.choices[0].message.content.strip().lower()
    if "option_1" in texto:
        return "option_1"
    elif "option_2" in texto:
        return "option_2"
    elif "option_3" in texto:
        return "option_3"
    else:
        return "option_1"

# Pruebo con los primeros 20 ejemplos para no gastar limite de API
df_test = df_eval.head(20).copy()
df_test["pred_llm"] = df_test.apply(baseline_llm_zeroshot, axis=1)
acc_llm = (df_test["pred_llm"] == df_test["correct"]).mean()

print(f"Baseline LLM zero-shot (20 ejemplos): {acc_llm:.3f} ({acc_llm*100:.1f}%)")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmra6gcpf4vbbzx0qf02b3jz` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99860, Requested 524. Please try again in 5m31.776s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Interesante. Con solo 20 ejemplos el LLM zero-shot da 35%, que es similar a los baselines simples.

In [ ]:
# Pruebo con 50 ejemplos
df_test50 = df_eval.head(50).copy()
df_test50["pred_llm"] = df_test50.apply(baseline_llm_zeroshot, axis=1)
acc_llm50 = (df_test50["pred_llm"] == df_test50["correct"]).mean()

print("RESULTADOS COMPARATIVOS")
print("=" * 40)
print(f"Baseline aleatorio:           28.1%")
print(f"Baseline respuesta mas larga: 37.6%")
print(f"Baseline coincidencia lexica: 38.0%")
print(f"LLM zero-shot (50 ejemplos):  {acc_llm50*100:.1f}%")

Muy interesante. El LLM zero-shot (36%) no mejora significativamente los baselines simples (38%). Esto es un hallazgo importante para tu TFG.
Pero hay que tener en cuenta algo: el dataset está en inglés y el LLM está respondiendo en inglés. Cuando lleguen los datos reales de la competición (en español), los resultados pueden ser diferentes.
